# Getting Data from SkyMapper survey using TAP
General information on Skymapper: [https://skymapper.anu.edu.au/about-skymapper/](https://skymapper.anu.edu.au/about-skymapper/)  . This page gives information on the meaning of the various columns: [https://skymapper.anu.edu.au/table-browser/](https://skymapper.anu.edu.au/table-browser/)
To access the date, we will use TAP (Table Access Protocol), which is a table query based upon standards from the Virtual Observatory



In [1]:
import pyvo as vo
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.table import Table
from regions import CircleSkyRegion
from regions import Regions

myra = str(200.67219680842)
mydec = str(-36.87681628081452)


#create the TAP query. Note that the search radius is in degrees, and 0.0833 is 5 arcminutes
my_tap_query = ("SELECT DISTANCE(POINT('ICRS', raj2000, dej2000), POINT('ICRS', "   
              + myra + ", " + mydec + 
              ")) AS dist, "  + 
              "m.object_id,m.raj2000,m.dej2000,m.g_flags,m.g_nimaflags,m.i_flags,m.i_nimaflags, " + 
            "m.g_psf,m.e_g_psf,m.i_psf,m.e_i_psf  FROM dr2.master AS m " + 
              "WHERE 1 = CONTAINS(POINT('ICRS', raj2000, dej2000), " 
              "CIRCLE('ICRS',"  + 
               myra +", "+ mydec + 
              ", 0.083 )) ORDER BY dist " )
              
 

print(my_tap_query)



SELECT DISTANCE(POINT('ICRS', raj2000, dej2000), POINT('ICRS', 200.67219680842, -36.87681628081452)) AS dist, m.object_id,m.raj2000,m.dej2000,m.g_flags,m.g_nimaflags,m.i_flags,m.i_nimaflags, m.g_psf,m.e_g_psf,m.i_psf,m.e_i_psf  FROM dr2.master AS m WHERE 1 = CONTAINS(POINT('ICRS', raj2000, dej2000), CIRCLE('ICRS',200.67219680842, -36.87681628081452, 0.083 )) ORDER BY dist 


In [2]:
tap_service = vo.dal.TAPService("https://api.skymapper.nci.org.au/public/tap/")
#tap_results =  tap_service.search("SELECT DISTANCE(POINT('ICRS', raj2000, dej2000), POINT('ICRS', 200.67219680842, -36.87681628081452)) AS dist, m.object_id,m.raj2000,m.dej2000  FROM dr2.master AS m WHERE 1 = CONTAINS(POINT('ICRS', raj2000, dej2000), CIRCLE('ICRS',200.67219680842, -36.87681628081452, 0.0083 ))"  )
tap_results =  tap_service.search(my_tap_query)

The results of the TAP query are saved as a VO table.  Convert to a Pandas dataframe, which is easier to manipulate.

In [ ]:
astropy_table = tap_results.to_table()
df = astropy_table.to_pandas()
print(df)

## Get rid of stars with bad photometry
Keep only stars which havve good photometry in both g and i filters.  Good photometric measuresments  have flags and nimaflags = 0 and photometric errors < 0.02 mag in both the g and i filters. 

In [4]:
g_good1 = df['g_flags'] == 0 
g_good2 = df['g_nimaflags'] == 0
g_good3 = df['e_g_psf'] < 0.02
i_good1 = df['i_flags'] == 0 
i_good2 = df['i_nimaflags'] == 0
i_good3 = df['e_i_psf'] < 0.02

#combine these comparisons to make a new dataframe
df_good = df[g_good1 & g_good2 & g_good3 & i_good1 & i_good2 & i_good3]
num_good = df_good.shape[0]
print("Number of stars with good photometry: ",  num_good)


Number of stars with good photometry:  69


## Determine which telescope can observe this star

Find the number of bright stars (which will likely be observed with Mookodi and its 5 arcminut field of view) and the number visible in the SHOC field of view, which is 2.8 arcminutes on a side. 

In [5]:
#with its bigger field of view, Mookodi is better for bright stars
g_bright = df_good['g_psf'] < 16.5
i_bright = df_good['i_psf'] < 16.5
df_bright = df_good[g_bright & i_bright]
shoc_comp = df_good['dist'] < 0.045   #SHOC field of view is 2.8x2.8 arcminutes, so use 2.7 arcminutes
df_shoc = df_good[shoc_comp]
num_bright = df_bright.shape[0]
num_shoc_comp = df_shoc.shape[0]
if num_bright > 4 :
    mookodi = " and Mookodi can observe this star"
else : 
    mookodi = ""

        
print("Number of bright stars for Mookodi: ", num_bright, mookodi )
if num_shoc_comp > 4 :
    shoc = " and SHOC can observe this star"
else : 
    shoc = ""


print("Number of comparison stars for SHOC: ", num_shoc_comp, shoc)


Number of bright stars for Mookodi:  19  and Mookodi can observe this star
Number of comparison stars for SHOC:  12  and SHOC can observe this star


## Create a region file
Create a region file for use with ds9, with all of the SHOC and Mookodi stars stars, and of course our target star

In [6]:
f = open("region.reg", "w")
f.write("global color=blue dashlist=8 3 width=5\n")
f.write("fk5\n")

new_ra = str(df['raj2000'].iloc[0])
new_dec= str(df['dej2000'].iloc[0])
f.write("circle(" + new_ra + "," + new_dec + ", 0.0014)\n")

f.write("global color=green dashlist=8 3 width=3\n")

for index, row in df_bright.iterrows():
    new_ra = str(row['raj2000'])
    new_dec= str(row['dej2000'])
            
    f.write("circle(" + new_ra + "," + new_dec + ", 0.0014)\n")
    
for index, row in df_bright.iterrows():
    new_ra = str(row['raj2000'])
    new_dec= str(row['dej2000'])
            
    f.write("circle(" + new_ra + "," + new_dec + ", 0.0014)\n")

f.write("global color=red dashlist=8 3 width=2\n")
for index, row in df_shoc.iterrows():
    new_ra = str(row['raj2000'])
    new_dec= str(row['dej2000'])
            
    f.write("circle(" + new_ra + "," + new_dec + ", 0.0014)\n")


f.close()

#open and read the file after the overwriting:
f = open("region.reg", "r")
# print(f.read())

## Download an image of the field
We now want to get an image of the field containing our star (which is the first star in df), and mark the location of our star and the comparison stars.  To do so, we first need to use the object id to find the image ID and CCD number in the SkyMapper catalog.


In [7]:
my_object_id = str(df['object_id'].iloc[0])

my_tap_query = ("SELECT object_id, image_id, ccd, filter, fwhm FROM dr2.photometry WHERE object_id="
                + my_object_id + " AND filter='i' " )
print(my_tap_query)



SELECT object_id, image_id, ccd, filter, fwhm FROM dr2.photometry WHERE object_id=96670457 AND filter='i' 


We now can download the image.  Note that the above query will likley result in multiple images, and we are just going to download the firstimage.  After the file is downloaded, we save it to disk and also make it available to python.

In [8]:
tap_results =  tap_service.search(my_tap_query)
astropy_table = tap_results.to_table()
image = astropy_table.to_pandas()
print(image)
my_image_id= str(image['image_id'].iloc[0])
my_ccd_num = image['ccd'].iloc[0]
if my_ccd_num < 10 :
    my_ccd = "0" + str(my_ccd_num)
else :
    my_ccd = str[my_ccd_num]
    
print(my_ccd)
        

my_https_query = ("https://api.skymapper.nci.org.au/public/siap/dr2/get_image?IMAGE="
                 + my_image_id + "-"  + my_ccd + "&SIZE=0.167&POS="
                 + myra + "," + mydec + 
                 "&FORMAT=fits")
print(my_https_query)

import requests
from astropy.io import fits
from io import BytesIO

r=requests.get(my_https_query)
with open("output.fits", "wb") as f:
    f.write(r.content) 

with fits.open(BytesIO(r.content)) as hdul:
    header=hdul[0].header
    image_data=hdul[0].data
    
#print(header)

   object_id        image_id  ccd filter  fwhm
0   96670457  20180129161801    9      i  4.10
1   96670457  20150620112921    9      i  4.78
2   96670457  20150312135519    9      i  4.00
3   96670457  20150603081735    9      i  5.81
4   96670457  20150306135554   10      i  4.92
5   96670457  20170407115503    9      i  4.09
09
https://api.skymapper.nci.org.au/public/siap/dr2/get_image?IMAGE=20180129161801-09&SIZE=0.167&POS=200.67219680842,-36.87681628081452&FORMAT=fits


## Startup ds9 and circle the stars

In [ ]:


import os
ds9 = ("/Applications/SAOImageDS9.app/Contents/MacOS/ds9 "
     + "output.fits -region region.reg" )
os.system(ds9)